# 🛣️ RoadScan AI — Pothole Detection YOLOv8 Training
**Thesis: Pothole Detection Web App for LGUs**

This notebook trains a YOLOv8 model on the RDD2022 dataset (pothole class only).

**Steps:**
1. Upload `pothole_rdd2022.zip` (already filtered & converted to YOLO format)
2. Train YOLOv8n model on GPU
3. Evaluate results (target: mAP@50 ≥ 80%)
4. Download `best.pt` to use in the web app

## 1. Setup

In [ ]:
!pip install ultralytics -q
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Upload Dataset
Upload `pothole_rdd2022.zip` from your local `Prototype/ml/` folder.

In [ ]:
from google.colab import files
print("📁 Upload pothole_rdd2022.zip from your Prototype/ml/ folder...")
uploaded = files.upload()

In [ ]:
# Unzip the dataset
!unzip -q pothole_rdd2022.zip -d /content/

# Fix the path in data.yaml to point to Colab location
import yaml
yaml_path = '/content/pothole_rdd2022/data.yaml'
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data['path'] = '/content/pothole_rdd2022'
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("\n✅ Dataset ready!")
!echo "Train:" && ls /content/pothole_rdd2022/train/images/ | wc -l
!echo "Val:" && ls /content/pothole_rdd2022/val/images/ | wc -l
!echo "Test:" && ls /content/pothole_rdd2022/test/images/ | wc -l

## 3. Train YOLOv8

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/pothole_rdd2022/data.yaml',
    epochs=50,
    batch=16,
    imgsz=640,
    project='pothole_detection',
    name='train',
    exist_ok=True,
    # Optimizer
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    # Data augmentation (key for high accuracy)
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5,
    shear=2.0, flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.15,
    # Other
    patience=20,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
)

## 4. Evaluate Results

In [ ]:
# Validate
val_results = model.val(data='/content/pothole_rdd2022/data.yaml')

print("=" * 50)
print("📊 FINAL VALIDATION METRICS")
print("=" * 50)
print(f"  mAP@50:      {val_results.box.map50:.4f}")
print(f"  mAP@50-95:   {val_results.box.map:.4f}")
print(f"  Precision:    {val_results.box.mp:.4f}")
print(f"  Recall:       {val_results.box.mr:.4f}")
print("=" * 50)

if val_results.box.map50 >= 0.80:
    print("🎉 TARGET ACHIEVED: mAP@50 ≥ 80%!")
else:
    print(f"⚠️  mAP@50 = {val_results.box.map50:.1%}, below 80% target.")
    print("  Try: more epochs, larger model (yolov8s), or more augmentation.")

In [ ]:
# Show training curves
from IPython.display import Image, display
display(Image(filename='pothole_detection/train/results.png', width=800))

In [ ]:
# Show sample predictions on validation images
display(Image(filename='pothole_detection/train/val_batch0_pred.jpg', width=800))

## 5. Download Trained Model
Download `best.pt` and place it in your project at:
```
Prototype/ml/models/best.pt
```
The web app backend will automatically use it.

In [ ]:
from google.colab import files
files.download('pothole_detection/train/weights/best.pt')